In [1]:
import cvxpy as cp
import numpy as np
from sympy.physics.quantum import TensorProduct
from qiskit.quantum_info import  DensityMatrix

# Theoretical Framework

Consider the standard Bell scenario in which Alice and Bob freely choose between two possible measurements each:

* Alice can measure two observables, $\hat{A}_0$ and $\hat{A}_1$.
* Bob can measure two observables, $\hat{B}_0$ and $\hat{B}_1$.

Each measurement produces a classical outcome $\pm 1$. Our objective is to determine the maximum possible violation of the CHSH inequality using a semidefinite programming (SDP) problem.

## Construction of the moment matrix

Considering the operators:

$$
\hat{A_0} = \hat{\Pi}_{a=+1}^{A_0} - \hat{\Pi}_{a=-1}^{A_0} \\
\hat{A_1} = \hat{\Pi}_{a=+1}^{A_1} - \hat{\Pi}_{a=-1}^{A_1} \\
\hat{B_0} = \hat{\Pi}_{b=+1}^{B_0} - \hat{\Pi}_{b=-1}^{B_0} \\
\hat{B_1} = \hat{\Pi}_{b=+1}^{B_1} - \hat{\Pi}_{b=-1}^{B_1},
$$

where the operator $\hat{\Pi}_{m}^{M}$ is the projector associated with the measurement $\hat{M}$ and classical outcome $m$.

Note that the operators $\hat{A}_i$ and $\hat{B}_j$ are Hermitian ($\hat{A}_i = \hat{A}_i^\dagger$ and $\hat{B}_j = \hat{B}_j^\dagger$).

We can define the set of operators:

$$\mathcal{F} = \{ \hat{I}, \hat{A}_0, \hat{A}_1, \hat{B}_0, \hat{B}_1 \}.$$

From these, we define the moment matrix $\Gamma$, with entries:

$$\Gamma_{ij} = \mathrm{Tr} \big( \rho \, \mathcal{F}_i^\dagger \mathcal{F}_j \big),$$

for an arbitrary quantum state $\rho$.

Let's calculate some of the matrix entries:

* Normalization of $\rho$:
    $$\Gamma_{00} = \mathrm{Tr}(\rho \, \hat{I}) = \mathrm{Tr}(\rho) = 1.$$

* Expected value of an Alice observable:
    $$\Gamma_{01} = \mathrm{Tr}(\rho \, \hat{A}_0) = \langle \hat{A}_0 \rangle.$$

* Expected value of the square of an Alice observable:
    $$\Gamma_{11} = \mathrm{Tr}(\rho \, \hat{A}_0^{\dagger} \hat{A}_0) = \mathrm{Tr}(\rho \, \hat{A}_0 \hat{A}_0) = \mathrm{Tr}(\rho \, \hat{I}) = 1.$$

* Alice–Bob correlations:
    $$\Gamma_{13} = \mathrm{Tr}(\rho \, \hat{A}_0^{\dagger} \hat{B}_0) = \langle \hat{A}_0 \hat{B}_0 \rangle.$$

The matrix $\Gamma$ is **Hermitian**, which reduces the number of independent entries in the matrix. For example:

$$\Gamma_{31} = \langle \hat{B}_0^{\dagger} \hat{A}_0 \rangle = \langle \hat{A}_0 \hat{B}_0^{\dagger} \rangle = \langle \hat{A}_0^{\dagger} \hat{B}_0 \rangle = \Gamma_{13}.$$

The terms of the type $\langle \hat{A}_i \hat{A}_j \rangle$ and $\langle \hat{B}_i \hat{B}_j \rangle$, for $i \neq j$, are left as free variables $v_{ij}$, since they do not correspond to any measurable value in the proposed Bell scenario.

Thus, the moment matrix associated with our particular $\mathcal{F}$ is written as:

$$
\Gamma =
\begin{pmatrix}
1 & \langle \hat{A}_0 \rangle & \langle \hat{A}_1 \rangle & \langle \hat{B}_0 \rangle & \langle \hat{B}_1 \rangle \\
\langle \hat{A}_0 \rangle & 1 & v_{12} & \langle \hat{A}_0 \hat{B}_0 \rangle & \langle \hat{A}_0 \hat{B}_1 \rangle \\
\langle \hat{A}_1 \rangle & v_{12} & 1 & \langle \hat{A}_1 \hat{B}_0 \rangle & \langle \hat{A}_1 \hat{B}_1 \rangle \\
\langle \hat{B}_0 \rangle & \langle \hat{A}_0 \hat{B}_0 \rangle & \langle \hat{A}_1 \hat{B}_0 \rangle & 1 & v_{34} \\
\langle \hat{B}_1 \rangle & \langle \hat{A}_0 \hat{B}_1 \rangle & \langle \hat{A}_1 \hat{B}_1 \rangle & v_{34} & 1
\end{pmatrix}.
$$

In the language of SDP, it is common to show only the upper triangular part:

$$
\Gamma =
\begin{pmatrix}
1 & \langle \hat{A}_0 \rangle & \langle \hat{A}_1 \rangle & \langle \hat{B}_0 \rangle & \langle \hat{B}_1 \rangle \\
& 1 & v_{12} & \langle \hat{A}_0 \hat{B}_0 \rangle & \langle \hat{A}_0 \hat{B}_1 \rangle \\
& & 1 & \langle \hat{A}_1 \hat{B}_0 \rangle & \langle \hat{A}_1 \hat{B}_1 \rangle \\
& & & 1 & v_{34} \\
& & & & 1
\end{pmatrix}.
$$

For the behavior $\mathcal{P}$ we are analyzing to be part of the set of behaviors $Q_{\mathcal{F}}$ achievable by a state $\rho$, the matrix $\Gamma$ must be positive semidefinite:

$$
\Gamma \succeq 0.
$$

We can impose additional conditions over which we want to optimize. If we define the CHSH operator as:

$$
\hat{S} = \hat{A}_0 \hat{B}_0 + \hat{A}_0 \hat{B}_1 + \hat{A}_1 \hat{B}_0 - \hat{A}_1 \hat{B}_1.
$$

The value of the CHSH inequality can be written as:
$$
S = \mathrm{Tr}(\rho \, \hat{S})
= \langle \hat{A}_0 \hat{B}_0 \rangle
+ \langle \hat{A}_0 \hat{B}_1 \rangle
+ \langle \hat{A}_1 \hat{B}_0 \rangle
- \langle \hat{A}_1 \hat{B}_1 \rangle.
$$

Thus, our objective is to find the maximum expected value of this operator subject to the condition $\Gamma \succeq 0$. In other words, we want to find the maximum possible value such that the behavior still belongs to $Q_{\mathcal{F}}$.

This defines a convex optimization problem that can be solved using SDP. The optimization problem is formulated as follows:

$$
\max \quad S = \langle \hat{A}_0 \hat{B}_0 \rangle
+ \langle \hat{A}_0 \hat{B}_1 \rangle
+ \langle \hat{A}_1 \hat{B}_0 \rangle
- \langle \hat{A}_1 \hat{B}_1 \rangle \\
\text{subject to} \quad \Gamma \succeq 0 \quad (\mathcal{P} \in Q_{\mathcal{F}}).
$$

# Code

We create the moment matrix $\Gamma$ using the `cvxpy` library. Except the entry $\Gamma_{00} = 1$ and the entries $\Gamma_{ii} = \langle \hat{I} \rangle = 1$, the other entries are left as variables to be optimized.

In [2]:
Γ_00 = 1                # <I>
Γ_01 = cp.Variable()    # <I*A0>  = <A0>
Γ_02 = cp.Variable()    # <I*A1>  = <A1>
Γ_03 = cp.Variable()    # <I*B0>  = <B0>
Γ_04 = cp.Variable()    # <I*B1>  = <B1>
Γ_11 = 1                # <A0*A0> = <I> = 1
Γ_12 = cp.Variable()    # <A0*A1> = v_12
Γ_13 = cp.Variable()    # <A0*B0>
Γ_14 = cp.Variable()    # <A0*B1>
Γ_22 = 1                # <A1*A1> = <I> = 1
Γ_23 = cp.Variable()    # <A1*B0>
Γ_24 = cp.Variable()    # <A1*B1>
Γ_33 = 1                # <B0*B0> = <I> = 1
Γ_34 = cp.Variable()    # <B0*B1> = v_34
Γ_44 = 1                # <B1*B1> = <I> = 1


Γ = cp.bmat([[Γ_00, Γ_01, Γ_02, Γ_03, Γ_04],
             [Γ_01, Γ_11, Γ_12, Γ_13, Γ_14],
             [Γ_02, Γ_12, Γ_22, Γ_23, Γ_24],
             [Γ_03, Γ_13, Γ_23, Γ_33, Γ_34],
             [Γ_04, Γ_14, Γ_24, Γ_34, Γ_44]])      

The condition is that the moment matrix must be positive semidefinite ($\Gamma \succeq 0$).

In [3]:
constraints = [

    Γ >> 0,
]

The optimization objective is defined as the expected value of $S$.

In [4]:
chsh_expression = Γ_13 + Γ_14 + Γ_23 - Γ_24 # CHSH expression: <A0*B0> + <A0*B1> + <A1*B0> - <A1*B1> 
objective = cp.Maximize(chsh_expression)

The problem is solved

$$
\max \quad S = \langle \hat{A}_0 \hat{B}_0 \rangle
+ \langle \hat{A}_0 \hat{B}_1 \rangle
+ \langle \hat{A}_1 \hat{B}_0 \rangle
- \langle \hat{A}_1 \hat{B}_1 \rangle, \\
\text{subject to} \quad  \Gamma \succeq 0 \quad [\mathcal{P} \in Q_{\mathcal{F}}].  \
$$

In [5]:
problem = cp.Problem(objective, constraints)
problem.solve();

In [6]:
print(f"The program status is: {problem.status}")
print(f"The maximum violation found for CHSH is: {problem.value:.8f}")
print("\nOptimal Γ:")

DensityMatrix(np.round(Γ.value, 4)).draw(output = "latex")

The program status is: optimal
The maximum violation found for CHSH is: 2.82841658

Optimal Γ:


<IPython.core.display.Latex object>

We find that the maximum value $S$ can take such that $\mathcal{P}$ still belongs to $Q_{\mathcal{F}}$ is $2.82841658 \approx 2 \sqrt{2}$. However, to guarantee that this is indeed the maximum possible quantum violation of the inequality, we must find a quantum state and a set of quantum measurements capable of reproducing this behavior.

# Verification that Bell states reproduce this behavior

Let us use the Bell state $\ket{\phi^+}\bra{\phi^+}$

In [7]:
phi_m =  np.array([[1,0,0,1],
                   [0,0, 0,0],
                   [0,0,0,0],
                   [1,0,0,1]])/2

and the measurement operators:


$\hat{A_0} = \hat{\sigma_z}$

$\hat{A_1} = \hat{\sigma_x}$

$\hat{B_0} = \frac{1}{\sqrt{2}}(\hat{\sigma_x} + \hat{\sigma_z})$

$\hat{B_1} = \frac{1}{\sqrt{2}}(-\hat{\sigma_x} + \hat{\sigma_z})$

In [8]:
identity2 = np.array([[1,0],[0,1]])
pauliX = np.array([[0,1],[1,0]])
pauliZ = np.array([[1,0],[0,-1]])

B0_op = (pauliX  + pauliZ)/np.sqrt(2) 
B1_op = (-pauliX  + pauliZ)/np.sqrt(2)

We calculate the individual expected values. For example:

$$ \mathrm{Tr}(\rho \, \hat{A}_0) = \langle \hat{A}_0 \rangle$$

In [9]:
A0 = np.trace(TensorProduct(pauliZ, identity2) @ phi_m) 
A1 = np.trace(TensorProduct(pauliX, identity2) @ phi_m) 

B0 = np.trace(TensorProduct(identity2,B0_op) @ phi_m) 
B1 = np.trace(TensorProduct(identity2,B1_op) @ phi_m) 

A0, A1, B0, B1

(np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0))

We calculate the joint expected values. For example:

$$ \mathrm{Tr}(\rho \, \hat{A}_0 \hat{B}_0) = \langle \hat{A}_0 \hat{B}_0  \rangle$$

In [10]:
A0_B0 = np.trace(np.dot(phi_m,TensorProduct(pauliZ, B0_op))) 
A0_B1 = np.trace(np.dot(phi_m,TensorProduct(pauliZ, B1_op))) 

A1_B0 = np.trace(np.dot(phi_m,TensorProduct(pauliX, B0_op))) 
A1_B1 = np.trace(np.dot(phi_m,TensorProduct(pauliX, B1_op))) 



A0_B0, A0_B1, A1_B0, A1_B1

(np.float64(0.7071067811865475),
 np.float64(0.7071067811865475),
 np.float64(0.7071067811865475),
 np.float64(-0.7071067811865475))

We calculate the squared expected values. For example:

$$\mathrm{Tr}(\rho \, \hat{A}_0 \hat{A}_0) = \langle \hat{A}_0 \hat{A}_0  \rangle = \langle \hat{A}_0 ^2   \rangle = 1$$

In [11]:
A0_A0 = np.trace(np.dot(phi_m,TensorProduct(pauliZ,pauliZ))) 
A1_A1 = np.trace(np.dot(phi_m,TensorProduct(pauliX,pauliX)))

B0_B0 = np.trace(np.dot(phi_m,TensorProduct(B0_op,B0_op)))
B1_B1 = np.trace(np.dot(phi_m,TensorProduct(B1_op,B1_op)))

A0_A0, A1_A1, B0_B0, B1_B1

(np.float64(1.0),
 np.float64(1.0),
 np.float64(0.9999999999999998),
 np.float64(0.9999999999999998))

In [12]:
Γ_Bell = np.array([[1, A0, A1, B0, B1],
                       [A0, A0_A0, 0, A0_B0, A0_B1],
                       [A1, 0, A1_A1, A1_B0, A1_B1],
                       [B0, A0_B0, A1_B0, B0_B0, 0],
                       [B1, A0_B1, A1_B1, 0, B1_B1]])


DensityMatrix(np.round(Γ_Bell, 4)).draw(output = "latex")

<IPython.core.display.Latex object>

We see that this state and the chosen measurements reproduce the behavior found with the SDP. Therefore, the value we found is indeed the maximum possible value of the CHSH expression within quantum theory, and it is achieved using Bell states.